In [1]:
import numpy as np
from scipy.stats import norm
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import h5py

In [2]:
# step1. generate parameter combinations
#1) BS
def generate_BS_params(n_sets, seed=None):
    if seed is not None:
        np.random.seed(seed)

    S0 = np.random.normal(loc=1.0, scale=0.2)  # S0 ~ N(1, 0.2^2) Greek 계산시 delta를 구할 경우 1로 안둠.
    r = np.random.uniform(0, 0.1, n_sets)      # r ~ U(0, 0.1)
    sigma = np.random.uniform(0.001, 1, n_sets)# σ ~ U(0.001, 1)
    T = np.random.uniform(0.1, 3, n_sets)      # T ~ U(0.1, 3)
    K = 1                                      # K = 1

    params = np.stack([S0, K, r, sigma, T], axis=1)
    return params

#2) heston
def generate_heston_params(n_sets, seed=None):
    if seed is not None:
        np.random.seed(seed)

    r = np.random.uniform(0, 0.1, n_sets)      # r ~ U(0, 0.1)
    lamb = np.random.beta(2, 18, n_sets) * 20  # λ ~ Beta(2, 18) × 20
    v_bar = np.random.beta(1, 19, n_sets)      # v_bar ~ Beta(1, 19)
    epsilon = np.random.uniform(0.1, 1, n_sets)# ξ ~ U(0.1, 1)
    rho = np.random.uniform(-1, 0, n_sets)     # ρ ~ U(-1, 0)
    Y0 = np.random.beta(1, 19, n_sets)         # Y₀ ~ Beta(1, 19)
    T = np.random.uniform(0.1, 3, n_sets)      # T ~ U(0.1, 3)
        
    params = np.stack([r, lamb, v_bar, epsilon, rho, Y0, T], axis=1)
    return params

def filter_milestein(params): # milestein condition
    _, lamb, v_bar, epsilon, *_ = params.T
    mask = 4 * lamb * v_bar > epsilon**2
    return params[mask]

def generate_valid_params(n_sets, seed=None):
    raw = generate_heston_params(int(n_sets * 1.1), seed)
    filtered = filter_milestein(raw)
    
    while len(filtered) < n_sets:
        extra = generate_heston_params(n_sets)
        extra = filter_milestein(extra)
        filtered = np.vstack([filtered, extra])
    
    return filtered[:n_sets]

def min_max_normalize(data):
    return (data - data.min()) / (data.max() - data.min())

In [ ]:
#step1-1
BS_paras = generate_BS_params(n_sets=100*(2**16), seed=1234)
BS_paras_scaled = BS_paras.copy()

for col in range(7):
    if col == 0:
        BS_paras_scaled[:, col] = (BS_paras[:, col] - 1.0) / 0.2
    else:
        BS_paras_scaled[:, col] = min_max_normalize(BS_paras[:, col])

In [ ]:
#step1-2 
Hes_paras = generate_valid_params(n_sets=100*(2**16), seed=1234) # 4.1초
Hes_paras_scaled = Hes_paras.copy()

for col in range(7):
    Hes_paras_scaled[:, col] = min_max_normalize(Hes_paras[:, col])

"""
print(f"\n파라미터 범위 확인:")
names = ['r', 'λ', 'v_bar', 'ξ', 'ρ', 'Y₀', 'T']
for i, name in enumerate(names):
    print(f"{name}: min={params[:,i].min():.4f}, max={params[:,i].max():.4f}, mean={params[:,i].mean():.4f}")
"""

'\nprint(f"\n파라미터 범위 확인:")\nnames = [\'r\', \'λ\', \'v_bar\', \'ξ\', \'ρ\', \'Y₀\', \'T\']\nfor i, name in enumerate(names):\n    print(f"{name}: min={params[:,i].min():.4f}, max={params[:,i].max():.4f}, mean={params[:,i].mean():.4f}")\n'

In [ ]:
# step2. generate path
#1) BS
def generate_BS_paths(eta, n_paths=2**10, dt=0.001):
    S0, K, r, sigma, T = eta
    n_steps = int(T / dt)

    W = np.random.randn(n_paths, n_steps)
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0

    for i in range(n_steps):
        S[:, i+1] = S[:, i] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * W[:, i])

    ST = S[:, -1]
    MT = S.min(axis=1)
    return ST, MT


#2) Heston
def generate_heston_paths(eta, n_paths=2**10, dt=0.001):
    r, lamb, v_bar, epsilon, rho, Y0, T = eta
    n_steps = int(T / dt)

    W1 = np.random.randn(n_paths, n_steps)
    W2 = np.random.randn(n_paths, n_steps)
    dWx = np.sqrt(dt) * W1
    dWy = np.sqrt(dt) * (rho * W1 + np.sqrt(1 - rho**2) * W2)

    X = np.zeros((n_paths, n_steps + 1))
    Y = np.zeros((n_paths, n_steps + 1))
    X[:, 0] = 0.0
    Y[:, 0] = Y0

    for i in range(n_steps):
        Y_t = np.maximum(Y[:, i], 0)
        X[:, i+1] = X[:, i] + (r - 0.5 * Y_t) * dt + np.sqrt(Y_t) * dWx[:, i]
        Y[:, i+1] = (Y_t
                     + lamb * (v_bar - Y_t) * dt
                     + epsilon * np.sqrt(Y_t) * dWy[:, i]
                     + 0.25 * epsilon**2 * (dWy[:, i]**2 - dt))

    XT = X[:, -1]
    YT = Y[:, -1]
    MT = X.min(axis=1)
    mask = filter_paths(XT, T)
    return XT, YT, MT, mask


# CPU 멀티프로세스
# N_WORKERS = os.cpu_count() # max : 32
N_WORKERS = 28

def _worker(args):
    eta, n_paths, dt = args
    return generate_heston_paths(eta, n_paths=n_paths, dt=dt)

def generate_heston_parallel(eta_batch, n_paths=2**10, dt=0.001):
    results = []
    # 이렇게 하면 계속 프로세스를 생성, 소멸 시켜 오버헤드가 큼
    # 이런건 멀티 쓰레드 방식
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor: 
        futures = {executor.submit(_worker, (eta, n_paths, dt)): eta
                   for eta in eta_batch} # excutor.map이 아니여서, 입력 순서 상관없이 완료된 순서대로 반환
        for future in as_completed(futures):
            eta = futures[future]
            XT, YT, MT, mask = future.result()
            results.append((eta, XT, YT, MT, mask))
    return results  

# 연율화 수익률 평균/분산 필터링 (배치 전체 기준)
def filter_paths(XT, T):
    annual_return = XT / T
    check_mean = np.abs(annual_return.mean()) <= 0.3
    check_var = annual_return.var() <= 1.0
    return bool(check_mean and check_var)

In [13]:
#step2-2
SAVE_PATH = "/mnt/d/heston_dataset.h5"
target = (2**16) * 100
true = 0
fail = 0
BATCH_SIZE = N_WORKERS * 4

with h5py.File(SAVE_PATH, "a") as f:
    if "data" not in f:
        f.create_dataset("data", shape=(0, 10), maxshape=(None, 10), # maxshape가 없으면 shape이 (0,10)로 고정.
                          dtype="float64", chunks=(10240, 10), compression="gzip") 

    for i in range(0, len(Hes_paras), BATCH_SIZE):
        eta_batch = Hes_paras[i:i+BATCH_SIZE]
        batch_results = generate_heston_parallel(eta_batch)

        for eta, XT, YT, MT, mask in batch_results:
            if mask:
                rows = np.column_stack([
                    np.tile(eta, (len(XT), 1)),
                    XT.reshape(-1, 1),
                    YT.reshape(-1, 1),
                    MT.reshape(-1, 1)
                ])
                f["data"].resize(f["data"].shape[0] + len(rows), axis=0)
                f["data"][-len(rows):] = rows
                true += 1
                if true % 20000 == 0:
                    print(f"[{true}/{target}] fail: {fail}")
            else:
                fail += 1

            if target <= true:
                break
        else:
            continue
        break
    else:
        Hes_paras = generate_valid_params(n_sets=(target - true), seed=1234)

print(f"done. true: {true}, fail: {fail}")

KeyboardInterrupt: 

In [ ]:
# closed-form
#1)vanilla
def BS_vanilla(eta, type='call'):
    # type : 'call' or 'put'
    S0, K, r, sigma, T = eta

    sqT  = sigma * T
    disc = np.exp(-r * T)

    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / sqT
    d2 = d1 - sqT

    if type == 'call':
        price = S0 * norm.cdf(d1) - K * disc * norm.cdf(d2)
    elif type == 'put':
        price = K * disc * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    return price



#2)down-and-out call
def BS_barrier(eta, type='call'):
    # type : 'call' or 'put'

    S0, K, r, sigma, T  = eta
    B = 0.8
    
    lam = r / sigma**2 + 0.5          
    sqT = sigma * np.sqrt(T)          
    disc = np.exp(-r * T)                     

    # vanilla
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / sqT
    d2 = d1 - sqT

    def vanilla_call():
        return S0 * norm.cdf(d1) - K * disc * norm.cdf(d2)

    def vanilla_put():
        return K * disc * norm.cdf(-d2) - S0 * norm.cdf(-d1)


    coef1 = S0 * (B / S0)**(2 * lam)           
    coef2 = K * disc * (B / S0)**(2 * lam - 2) 
    x1 = np.log(B / S0) / sqT + lam * sqT
    x2 = x1 - sqT

    if type == 'call':
        if B > K:
            raise ValueError("change B to be smaller than K")
        
        C_do = (vanilla_call() 
                - coef1 * norm.cdf(x1) 
                + coef2 * norm.cdf(x2))
        return C_do
        

    elif type == 'put':
        if B > K:
            raise ValueError("change B to be smaller than K")

        x_h1 = np.log(S0 / B) / sqT + lam * sqT
        x_h2 = x_h1 - sqT

        y1 = np.log(B**2 / (S0 * K)) / sqT + lam * sqT
        y2 = y1 - sqT

        P_do = (vanilla_put()
                + S0 * norm.cdf(-x_h1) - K * disc * norm.cdf(-x_h2)
                - coef1 * (norm.cdf(y1) - norm.cdf(x1))
                + coef2 * (norm.cdf(y2) - norm.cdf(x2)))
        return P_do

    else:
        raise ValueError("option_type must be 'call' or 'put'")